# 18 - The five ShakerMaker engines against SW4

The SW4 reference cases of `../../17_sw4_reference` (two media, three sources each, ~100 receivers per medium on
SW4 grid nodes, including 24 nodes of a DRM box) computed with every ShakerMaker engine:

| engine | what it is |
|---|---|
| `run` | direct engine, one Green's function per pair (not for the 256-subfault cases) |
| `exact` | `run_nearest` without reuse, one wavenumber step per pair of depths |
| `nearest` | `run_nearest` with reuse tolerances of 40 m, 5 m and 200 m |
| `inter_grid` | `run_inter`, grid scheme, cubic interpolation on a 40 m distance grid |
| `inter_tensor` | `run_inter`, tensor scheme (frequency cut, ZFP, decimated database) |

| case | medium | source |
|---|---|---|
| `loh1_p1`, `loh1_p2`, `loh1_f1` | SCEC LOH.1 | point at 2 km, point at 0.5 km, 256 subfaults (0.3-3.76 km) |
| `layered_p1`, `layered_p2`, `layered_f1` | layered 1D crust, 200 m at Vs 0.75 km/s on top | point at 6 km, point at 0.5 km, 256 subfaults |

Nothing is computed with the engines here: `../compare.py` reads the stored results (about 25 s) and this notebook
plots what it measures. Band of the comparison: 0.1-2 Hz.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run ../compare.py

sw4_rows = pd.DataFrame(rows_sw4)
eng_rows = pd.DataFrame(rows_eng)
drm_rows = pd.DataFrame(rows_drm)
colors = {"run": "k", "exact": "tab:green", "nearest": "#7f9fc6", "inter_grid": "tab:orange", "inter_tensor": "#c0392b"}
t = {m: np.arange(0.0, tmax[m], dt) for m in tmax}

least-squares amplitude factor ShakerMaker -> SW4, loh1 (check, not applied): 0.9704
least-squares amplitude factor ShakerMaker -> SW4, layered (check, not applied): 0.9958



ShakerMaker vs SW4: median / min correlation (best lag), median peak ratio, median nrms
loh1_p1  run          0.1-2.0  cc 1.000 / 0.307  peak 1.00  nrms 0.010  lag +0.000 s  (n=294)
loh1_p1  run          0.1-0.5  cc 1.000 / -0.668  peak 1.00  nrms 0.006  lag +0.000 s  (n=293)
loh1_p1  run          0.5-1.0  cc 1.000 / 0.052  peak 1.00  nrms 0.006  lag +0.000 s  (n=294)
loh1_p1  run          1.0-2.0  cc 1.000 / 0.686  peak 1.00  nrms 0.010  lag +0.000 s  (n=294)
loh1_p1  exact        0.1-2.0  cc 1.000 / 0.314  peak 1.00  nrms 0.007  lag +0.000 s  (n=294)
loh1_p1  exact        0.1-0.5  cc 1.000 / -0.638  peak 1.00  nrms 0.004  lag +0.000 s  (n=293)
loh1_p1  exact        0.5-1.0  cc 1.000 / 0.055  peak 1.00  nrms 0.005  lag +0.000 s  (n=294)
loh1_p1  exact        1.0-2.0  cc 1.000 / 0.677  peak 1.00  nrms 0.007  lag +0.000 s  (n=294)
loh1_p1  nearest      0.1-2.0  cc 1.000 / 0.307  peak 1.00  nrms 0.014  lag +0.000 s  (n=294)
loh1_p1  nearest      0.1-0.5  cc 1.000 / -0.668  peak 1.00  nr

## Agreement with SW4

Median correlation (best lag within 0.1 s) over every receiver and component with signal, 0.1-2 Hz. Missing cells:
engine not computed for that case (`run` takes hours on the 256-subfault cases).

In [2]:
sel = sw4_rows[sw4_rows.band == "0.1-2.0"]
cc = sel.groupby(["case", "engine"]).cc_best.median().unstack().reindex(index=cases, columns=engines)
peak = sel.groupby(["case", "engine"]).peak.median().unstack().reindex(index=cases, columns=engines)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
im = ax.imshow(cc.values, cmap="viridis", vmin=0.95, vmax=1.0, aspect="auto")
for i in range(len(cases)):
    for j in range(len(engines)):
        v = cc.values[i, j]
        ax.text(j, i, "-" if np.isnan(v) else f"{v:.3f}", ha="center", va="center", fontsize=8,
                color="w" if (not np.isnan(v) and v < 0.98) else "k")
ax.set_xticks(range(len(engines))); ax.set_xticklabels(engines, rotation=20)
ax.set_yticks(range(len(cases))); ax.set_yticklabels(cases)
fig.colorbar(im, ax=ax, label="median correlation with SW4")
fig.tight_layout()
fig.savefig("agreement_with_sw4.png", dpi=150, bbox_inches="tight")
print("median peak ratio ShakerMaker / SW4")
peak.round(3)

median peak ratio ShakerMaker / SW4


engine,run,exact,nearest,inter_grid,inter_tensor
case,,,,,
loh1_p1,0.996,0.996,0.996,0.996,0.990
loh1_p2,0.995,0.995,0.994,0.995,0.991
loh1_f1,NaN,1.000,1.001,1.000,0.999
layered_p1,0.992,0.992,0.993,0.992,0.991
layered_p2,0.989,0.989,0.990,0.989,0.987
layered_f1,NaN,0.998,0.999,0.998,0.998


## Traces at a surface receiver

Ring receiver 4 km from the epicentre. SW4 in the background; the five engines on top (they overlap: their
differences are much smaller than their difference with SW4).

In [3]:
rec = "ring_r4_a060"
fig, ax = plt.subplots(2, 3, figsize=(13, 5.5), sharex="row")
for r, case in enumerate(["loh1_f1", "layered_p2"]):
    m = case.split("_")[0]
    names, ref = sw4[case]
    k = names.index(rec)
    for j, comp in enumerate("ZEN"):
        ax[r, j].plot(t[m], band(ref[k, j], 0.1, 2.0), "--", color="#7f9fc6", lw=2.2, label="SW4")
        for e in engines:
            names_sm, u = sm[(case, e)]
            if u is not None:
                ax[r, j].plot(t[m], band(u[k, j], 0.1, 2.0), color=colors[e], lw=0.8, label=e)
        ax[r, j].set_title(f"{case} {rec} {comp}", fontsize=9)
    ax[r, 0].set_ylabel("displacement")
ax[0, 0].legend(fontsize=7)
for a in ax[-1]:
    a.set_xlabel("time (s)")
fig.tight_layout()
fig.savefig("traces_surface.png", dpi=150, bbox_inches="tight")

## The DRM box: difference between its two layers

What the DRM turns into forces is the difference between the outer and inner layers of the box, 50 m apart here.
Reuse (`nearest`) gives both layers the same or a jumping Green's function; the interpolated engines follow the
exact one.

In [4]:
case, o, i = "layered_p2", "drm_face_n_out", "drm_face_n_in"
names, ref = sw4[case]
io, ii = names.index(o), names.index(i)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for j, comp in enumerate("ZEN"):
    ax[j].plot(t["layered"], lowpass(ref[io, j] - ref[ii, j]), "--", color="#7f9fc6", lw=2.2, label="SW4")
    for e in ("run", "nearest", "inter_tensor"):
        u = sm[(case, e)][1]
        ax[j].plot(t["layered"], lowpass(u[io, j] - u[ii, j]), color=colors[e], lw=0.9, label=e)
    ax[j].set_title(f"{case}: {o} - {i}, {comp}", fontsize=9); ax[j].set_xlabel("time (s)")
ax[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig("drm_layer_difference.png", dpi=150, bbox_inches="tight")

d = drm_rows[drm_rows.ref != "sw4"].groupby(["case", "engine"]).nrms.median().unstack().reindex(index=cases)
d = d[[e for e in engines if e in d.columns]]
fig, ax = plt.subplots(figsize=(8, 3.4))
x, w = np.arange(len(cases)), 0.8 / len(d.columns)
for n, e in enumerate(d.columns):
    ax.bar(x + (n - (len(d.columns) - 1) / 2) * w, 100 * d[e].values, w, color=colors[e], label=e)
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(cases, rotation=15)
ax.set_ylabel("error of the layer difference\nagainst the exact engine (median, %)"); ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig("drm_layer_error.png", dpi=150, bbox_inches="tight")
(100 * d).round(3)

engine,exact,nearest,inter_grid,inter_tensor
case,,,,
loh1_p1,1.352,44.445,1.351,3.279
loh1_p2,0.519,64.863,0.519,2.923
loh1_f1,NaN,28.175,0.162,0.945
layered_p1,0.000,38.935,0.004,1.427
layered_p2,0.000,76.691,0.010,0.876
layered_f1,NaN,41.779,0.087,0.492


## Agreement against depth

The vertical profile 1 km from the epicentre crosses every interface and the source depth. Correlation with SW4 of
each receiver (worst component), `run` or `exact`. The receivers at or a few metres from a source depth depart from
SW4 with every engine: an open finding of the FK integration, documented in `../../17_sw4_reference`, not of the
engines.

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for a, m in zip(ax, ("loh1", "layered")):
    for case, mk in zip([f"{m}_p1", f"{m}_p2", f"{m}_f1"], "osd"):
        e = "run" if sm[(case, "run")][1] is not None else "exact"
        s = sel[(sel.case == case) & (sel.engine == e) & sel.receiver.str.startswith("prof_r1_")]
        worst = s.groupby("receiver").cc_best.min()
        z = [float(n.split("_z")[1]) / 1000 for n in worst.index]
        a.plot(worst.values, z, mk, ms=5, label=f"{case} ({e})")
    for zs in {"loh1": (2.0, 0.5), "layered": (6.0, 0.5)}[m]:
        a.axhline(zs, color="0.6", lw=0.8, ls=":")
    a.set_title(m); a.set_xlabel("correlation with SW4 (worst component)"); a.legend(fontsize=7)
ax[0].invert_yaxis(); ax[0].set_ylabel("receiver depth (km); dotted: point-source depths")
fig.tight_layout()
fig.savefig("agreement_vs_depth.png", dpi=150, bbox_inches="tight")

## Cost of each engine (wall time stored with each result, 32-core node)

In [6]:
wall = {}
for (case, e), _ in sm.items():
    for dd in sm_dirs:
        p = os.path.join(dd, f"{case}_{e}.npz")
        if os.path.exists(p):
            wall[(case, e)] = float(np.load(p)["wall_s"])
            break
pd.Series(wall).unstack().reindex(index=cases, columns=engines).round(1)

,run,exact,nearest,inter_grid,inter_tensor
loh1_p1,16.3,12.1,17.0,55.4,6.4
loh1_p2,51.7,36.0,48.0,139.6,12.5
loh1_f1,NaN,661.0,1650.8,1413.5,259.9
layered_p1,131.0,107.5,106.6,170.5,13.7
layered_p2,286.7,193.9,203.9,448.5,38.9
layered_f1,NaN,1377.6,914.6,3107.1,510.6


## Reading

- Every engine agrees with SW4 at the same level in each case: the differences with SW4 are those of the FK method
  against finite differences (and of the SW4 grid), not of the engine.
- Reuse (`nearest`) is fine for single traces but breaks the difference between the two layers of the DRM box;
  `inter_grid` and `inter_tensor` keep it within a few percent of the exact engine.
- Receivers at or a few metres from a source depth are an open finding of the FK integration (all engines).